# EDA — Home Credit Default Risk
Exploratory analysis to understand the training data before feature engineering.
Run with `--sample` during development; switch to full data for the final run.

In [ ]:
import sys
sys.path.insert(0, '..')  # make src/ importable from notebooks/

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from src.data import load_train, load_bureau, load_installments
from src.config import FIGURES_DIR

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
SAMPLE = True  # set False for full-data run
df = load_train(sample=SAMPLE)
print(f"Train shape: {df.shape}")

## 1. Target balance

In [ ]:
counts = df['TARGET'].value_counts()
rate   = df['TARGET'].mean()
print(f"Default rate: {rate:.2%}  ({counts[1]:,} defaults / {len(df):,} total)")

fig, ax = plt.subplots(figsize=(4, 3))
ax.bar(['Repaid (0)', 'Default (1)'], counts.values, color=['#3b6fff', '#e05252'], width=0.5)
ax.set_title('Target class balance', fontsize=11)
ax.set_ylabel('Count')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
for i, v in enumerate(counts.values):
    ax.text(i, v + len(df) * 0.005, f'{v/len(df):.1%}', ha='center', fontsize=9)
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'target_balance.png', dpi=150)
plt.show()
# Observation: strong class imbalance (~8% default); must use class weights or resampling.

## 2. Missing values

In [ ]:
miss = df.isnull().mean().sort_values(ascending=False)
miss_high = miss[miss > 0.30]  # columns missing >30 % of values
print(f"Columns with >30 % missing: {len(miss_high)}")

fig, ax = plt.subplots(figsize=(8, max(3, len(miss_high) * 0.25)))
miss_high.plot.barh(ax=ax, color='#e8922a')
ax.set_xlabel('Missing fraction')
ax.set_title('Columns with >30 % missing values', fontsize=11)
ax.xaxis.set_major_formatter(mticker.PercentFormatter(1))
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'missing_values.png', dpi=150)
plt.show()
# Decision note: high-missing columns will be imputed with median/mode in Phase 2;
# a 'was_missing' flag will be added for columns > 40 % missing (informative missingness).

## 3. Key numeric distributions

In [ ]:
num_cols = ['AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'DAYS_BIRTH', 'DAYS_EMPLOYED']
fig, axes = plt.subplots(1, len(num_cols), figsize=(14, 3))

for ax, col in zip(axes, num_cols):
    # Cap extreme outliers at 99th percentile for readability
    cap = df[col].quantile(0.99)
    data = df[col].clip(upper=cap).dropna()
    ax.hist(data, bins=40, color='#3b6fff', alpha=0.7, edgecolor='none')
    ax.set_title(col, fontsize=8)
    ax.tick_params(labelsize=7)

fig.suptitle('Key numeric feature distributions (capped at 99th pct)', fontsize=10)
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'numeric_distributions.png', dpi=150)
plt.show()
# DAYS_BIRTH is negative (days before application); convert to age in years in features.
# DAYS_EMPLOYED has anomalous value 365243 for pensioners — treat as missing.

## 4. Fairness-relevant columns — gender, age, region
These columns drive the Phase 6 fairness audit; document their distributions now.

In [ ]:
# 4a. Gender (CODE_GENDER)
gender = df['CODE_GENDER'].value_counts()
print("Gender distribution:")
print(gender.to_string())
print()
print("Default rate by gender:")
print(df.groupby('CODE_GENDER')['TARGET'].mean().map('{:.2%}'.format).to_string())
# Note: 'XNA' category exists — treat as missing / drop in fairness analysis.

In [ ]:
# 4b. Age — derived from DAYS_BIRTH (negative days relative to application date)
df['AGE_YEARS'] = (-df['DAYS_BIRTH'] / 365.25).round(1)
print(df['AGE_YEARS'].describe().to_string())

# Age buckets used in the fairness audit (Phase 6)
df['AGE_BUCKET'] = pd.cut(
    df['AGE_YEARS'],
    bins=[0, 30, 40, 50, 60, 100],
    labels=['<30', '30-40', '40-50', '50-60', '60+']
)
print("\nDefault rate by age bucket:")
print(df.groupby('AGE_BUCKET', observed=True)['TARGET'].mean().map('{:.2%}'.format).to_string())

In [ ]:
# 4c. Region (REGION_RATING_CLIENT — 1/2/3 risk rating of region)
print("Region rating distribution:")
print(df['REGION_RATING_CLIENT'].value_counts().sort_index().to_string())
print("\nDefault rate by region rating:")
print(df.groupby('REGION_RATING_CLIENT')['TARGET'].mean().map('{:.2%}'.format).to_string())
# Region is a proxy for socioeconomic status; note this for the fairness report.

## 5. Supplementary table shapes

In [ ]:
from src.data import load_bureau, load_installments, load_previous_applications

bureau   = load_bureau()
installs = load_installments()
prev_app = load_previous_applications()

print(f"bureau:                {bureau.shape}")
print(f"installments_payments: {installs.shape}")
print(f"previous_application:  {prev_app.shape}")
print()
print(f"Applicants in train:   {df['SK_ID_CURR'].nunique():,}")
print(f"Unique in bureau:      {bureau['SK_ID_CURR'].nunique():,}")
print(f"Unique in installs:    {installs['SK_ID_CURR'].nunique():,}")
# Coverage check: not all applicants have bureau / installment history — that's fine.

## 6. Summary notes for Phase 2

| Observation | Action in Phase 2 |
|---|---|
| ~8 % default rate | `class_weight='balanced'` in sklearn; `scale_pos_weight` in boosters |
| `DAYS_BIRTH` negative | Convert to `AGE_YEARS = -DAYS_BIRTH / 365.25` |
| `DAYS_EMPLOYED = 365243` anomaly | Replace with NaN before using |
| High-missing columns | Median/mode imputation + binary `_was_missing` flag |
| `CODE_GENDER == 'XNA'` | Map to NaN; exclude from fairness groups |
| Bureau / installs coverage | Applicants with no history → default to 0 / NaN for aggregates |